In [1]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import geopandas as gpd
import os
from dotenv import load_dotenv
from geoalchemy2.elements import WKBElement, WKTElement
load_dotenv()

True

### DATA BASE ENGINE

In [2]:
existing_database_url = os.getenv("BISELCO")
if existing_database_url:
    EXISTING_GIS_DATABASE= create_engine(existing_database_url)

current_database_url = os.getenv("BISELCOWEBSITE")
if current_database_url:
    CURRENT_GIS_DATABASE= create_engine(current_database_url)
    Session = sessionmaker(CURRENT_GIS_DATABASE)

In [9]:
# MUNICIPALITY DATA MIGRATION, NORMALIZATION AND PROCESSING
municipality = gpd.pd.read_sql(
    sql="""SELECT distinct municipality FROM gis.franchise_area order by municipality;""",
    con=EXISTING_GIS_DATABASE
)

values_mun = [(m,) for m in municipality['municipality'].to_list()]

# DATA MIGRATION FOR MUNICIPALITY VALUES
with Session.begin() as session:
    session.execute(statement=text(
        
        """INSERT INTO gis.municipality (name)
        VALUES (:mun)"""),
        params= [{"mun": m[0]} for m in values_mun] 
    )
    session.commit()
print("sucessfully inserted municipality")

sucessfully inserted municipality


In [10]:
# VILLAGE DATA MIGRATION, NORMALIZTION AND PROCESSING
village = gpd.pd.read_sql(
    sql = """SELECT distinct village, municipality FROM gis.franchise_area order by village;""",
    con=EXISTING_GIS_DATABASE
)

insert_stmt = text(
    """INSERT INTO gis.villages (municipality_id, name)
    VALUES (
        (SELECT id FROM gis.municipality WHERE name = :mun), :vill)"""
)
params = [{"mun": v[1], "vill": v[0]} for v in village.values]


with Session.begin() as session:
    session.execute(
        insert_stmt,
        params
    )
    session.commit()
print("sucessfully inserted village")

sucessfully inserted village


In [ ]:
# BOUNDARY DATA MIGRATION, NORMALIZATION AND PROCESSING

boundary = gpd.read_postgis(
    sql = """SELECT distinct geom, municipality, village FROM gis.franchise_area;""",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom"
)

params = [{"geom": b[0].wkt, "mun": b[1], "vill": b[2], "name": f"{b[2]} {b[1]}"} for b in boundary.values]

stmt = text(
    """INSERT INTO gis.boundary (geom, municipality_id, village_id, name)
    VALUES (:geom, (SELECT id FROM gis.municipality WHERE name = :mun), (SELECT id FROM gis.villages WHERE name = :vill and municipality_id = (SELECT id FROM gis.municipality WHERE name = :mun)), :name);
    """
)

with Session.begin() as session:
    session.execute(stmt, params)
    session.commit()
print("sucessfully inserted boundary")

sucessfully inserted boundary


#### SUBSTATION

In [36]:
substation = gpd.read_postgis(
    sql="""select * from gis.power_station;""",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom"
)
params = [{
    "geom": s[1].wkt,
    "substation_id": s[2],
    "phasing": s[4],
    "description": s[5],
    "vr": s[7],
    "vp": s[9],
    "active": True,
    "image": s[13]
} for s in substation.values]

print(params[0])
stmt = text("""
            INSERT INTO gis.substation (geom, substation_id, phasing, description,voltage_rating_kv, voltage_profile_id, is_active, image)
            values(:geom, :substation_id, :phasing, :description, :vr, :vp, :active, :image);
            """)

with Session.begin() as session:
    session.execute(stmt, params)
    session.commit()
print("sucessfully inserted substation")

{'geom': 'POINT (119.86627900000002 11.490639)', 'substation_id': 'NPCD0001', 'phasing': 'ABC', 'description': 'Linapacan Power Station-NPC', 'vr': 13.8, 'vp': 'VP0001', 'active': True, 'image': nan}
sucessfully inserted substation


#### BUS OR NODE

In [37]:
bus = gpd.read_postgis(
    sql="select * from gis.bus",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom"
)
params = [{
    "geom": b[1].wkt,
    "bus_id": b[2],
    "description": b[4],
    "nmv": b[5],
    "image": b[11],
    "is_active": True,

} for b in bus.values]


stmt = text("""
            INSERT INTO gis.bus (geom, bus_id, description, nominal_voltage_kv, image, is_active)
            values(:geom, :bus_id, :description, :nmv, :image, :is_active)
            ON CONFLICT (bus_id) DO UPDATE SET geom=EXCLUDED.geom, description=EXCLUDED.description, nominal_voltage_kv=EXCLUDED.nominal_voltage_kv, image=EXCLUDED.image, is_active=EXCLUDED.is_active;
            """)

with Session.begin() as session:
    session.execute(stmt, params)
    session.commit()
print("sucessfully inserted bus")


sucessfully inserted bus


#### PRIMARY LINES

In [ ]:
pl = gpd.read_postgis(
    sql="""WITH RECURSIVE feeder_tree AS (

    -- Root lines (connected to power station)
    SELECT 
        pl.geom,
        pl.primary_line_id,
        pl.phasing,
		pl.from_bus_id,
		pl.to_bus_id,
        pl.description,
        pl.configuration,
        pl.system_grounding_type,
        
        1 as level
    FROM gis.primary_line pl
    JOIN gis.power_station p
    ON ST_Intersects(ST_StartPoint(pl.geom), p.geom)

    UNION ALL

    -- Next connected lines
    SELECT
        pl2.geom,
        pl2.primary_line_id,
        pl2.phasing,
		pl2.from_bus_id,
		pl2.to_bus_id,
        pl2.description,
        pl2.configuration,
        pl2.system_grounding_type,
        
        ft.level + 1
    FROM feeder_tree ft
    JOIN gis.primary_line pl2
    ON ft.to_bus_id = pl2.from_bus_id
), primary_lines_ordered as (
SELECT distinct on (primary_line_id) geom ,primary_line_id,phasing, from_bus_id, to_bus_id,   description, configuration, system_grounding_type, level
FROM feeder_tree
ORDER BY  primary_line_id, level)
select * from primary_lines_ordered
order by level;""",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom",
)


stmt = text(
    """
            INSERT INTO gis.primary_lines 
            (geom, primary_line_id, phasing, description, configuration, system_ground_type, is_active)
            values(:geom, :pl_id, :phasing, :desc, :config, :ground_type, :is_active)
            ON CONFLICT (primary_line_id)
            DO UPDATE
            SET geom=EXCLUDED.geom, phasing=EXCLUDED.phasing, description=EXCLUDED.description, configuration=EXCLUDED.configuration, system_ground_type=EXCLUDED.system_ground_type, is_active=EXCLUDED.is_active;
            """
)

with Session.begin() as session:

    for p in pl.values:
        params = {
            "geom": p[0].wkt,
            "pl_id": p[1],
            "phasing": p[2],
            "desc": p[5],
            "config": p[6],
            "ground_type": p[7],
            "is_active": True,
        }
        try:
            session.execute(stmt, params)
            session.commit()
            print("sucessfully inserted primary lines", p[1])
        except Exception as e:
            session.rollback()
            print(f"❌ Failed on {p[1]}")
            print(e)
            break
        

sucessfully inserted primary lines


InvalidRequestError: Can't operate on closed transaction inside context manager.  Please complete the context manager before emitting further commands.

In [12]:
primary_lines = gpd.read_postgis(
    sql="select * from gis.primary_line",
    con=EXISTING_GIS_DATABASE
)
for i, row in primary_lines.iterrows():
    if row['primary_line_id'] not in pl_ids:
        print(row['primary_line_id'])